# Device and Browser Usage Analysis

## Project Overview
Analyze web/app usage by device, browser, OS, and version to understand compatibility and audience distribution.

## Learning Objectives
- Analyze device, browser, and OS distribution across web traffic
- Track adoption curves for new browser/OS versions
- Identify compatibility risk from legacy browser usage
- Compare engagement metrics across device categories
- Detect mobile vs desktop usage trends over time

## Problem Statement
A product team needs to decide which browsers and OS versions to support. Dropping legacy support saves engineering effort but risks losing users. Data-driven compatibility decisions require understanding the actual audience distribution.

## Why This Project Matters
Supporting every browser/OS combination is expensive. Analytics-driven support matrices ensure engineering effort goes where users actually are, while safely deprecating platforms with negligible traffic.

## Dataset Overview
Synthetic web analytics data: ~15,000 sessions over 6 months with device type, browser, OS, screen resolution, and engagement metrics.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 15000
dates = pd.date_range('2024-01-01', periods=180, freq='D')

device_types = ['desktop', 'mobile', 'tablet']
# Mobile share growing over time
base_device_w = np.array([0.42, 0.48, 0.10])

browsers = {
    'desktop': {'Chrome': 0.55, 'Safari': 0.18, 'Firefox': 0.12, 'Edge': 0.12, 'IE 11': 0.03},
    'mobile': {'Chrome Mobile': 0.45, 'Safari Mobile': 0.42, 'Samsung Internet': 0.08, 'Firefox Mobile': 0.05},
    'tablet': {'Safari Mobile': 0.55, 'Chrome': 0.30, 'Samsung Internet': 0.10, 'Firefox': 0.05},
}

os_map = {
    'desktop': {'Windows 11': 0.40, 'Windows 10': 0.25, 'macOS 14': 0.20, 'macOS 13': 0.08, 'Linux': 0.07},
    'mobile': {'iOS 17': 0.38, 'iOS 16': 0.12, 'Android 14': 0.30, 'Android 13': 0.15, 'Android 12': 0.05},
    'tablet': {'iPadOS 17': 0.50, 'iPadOS 16': 0.15, 'Android 14': 0.20, 'Android 13': 0.15},
}

resolutions = {
    'desktop': ['1920x1080', '2560x1440', '1366x768', '3840x2160'],
    'mobile': ['390x844', '412x915', '360x800', '414x896'],
    'tablet': ['1024x768', '1180x820', '800x1280'],
}

rows = []
for _ in range(n):
    date = np.random.choice(dates)
    day_idx = (date - dates[0]).days
    # Shift mobile share up over time
    dw = base_device_w.copy()
    dw[1] += 0.0003 * day_idx  # mobile grows
    dw[0] -= 0.0003 * day_idx  # desktop shrinks
    dw = np.clip(dw, 0.05, 0.7)
    dw /= dw.sum()

    device = np.random.choice(device_types, p=dw)
    b_names = list(browsers[device].keys())
    b_probs = list(browsers[device].values())
    browser = np.random.choice(b_names, p=b_probs)

    o_names = list(os_map[device].keys())
    o_probs = list(os_map[device].values())
    os_ver = np.random.choice(o_names, p=o_probs)

    resolution = np.random.choice(resolutions[device])

    # Engagement: desktop > tablet > mobile
    dev_mult = {'desktop': 1.3, 'mobile': 0.75, 'tablet': 1.0}[device]
    pages = max(1, int(np.random.poisson(4) * dev_mult))
    duration = max(5, np.random.lognormal(4.5, 1.0) * dev_mult)
    duration = min(duration, 3600)
    bounce = 1 if duration < 15 else 0

    rows.append({
        'date': date, 'device_type': device, 'browser': browser,
        'os': os_ver, 'resolution': resolution,
        'pages_viewed': pages, 'duration_sec': round(duration, 1), 'bounce': bounce
    })

df = pd.DataFrame(rows)
df['month'] = df['date'].dt.to_period('M')
print(f'Dataset: {df.shape}')
print(f'\nDevice distribution:\n{df["device_type"].value_counts()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nBrowser distribution:\n{df["browser"].value_counts()}')
print(f'\nOS distribution:\n{df["os"].value_counts()}')

## Device Type Distribution and Trend

In [ ]:
monthly_device = df.groupby(['month', 'device_type']).size().unstack(fill_value=0)
monthly_device_pct = monthly_device.div(monthly_device.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['device_type'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Device Type Share')
axes[0].set_ylabel('')

monthly_device_pct.index = monthly_device_pct.index.astype(str)
monthly_device_pct.plot(ax=axes[1], marker='o')
axes[1].set_title('Device Share Trend')
axes[1].set_ylabel('% of Sessions')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Browser Market Share

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
browser_counts = df['browser'].value_counts()
browser_counts.plot.barh(ax=ax, edgecolor='black')
ax.set_title('Sessions by Browser')
ax.set_xlabel('Sessions')
ax.invert_yaxis()
# Add percentage labels
total = len(df)
for i, (name, val) in enumerate(browser_counts.items()):
    ax.text(val + 50, i, f'{val/total*100:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'browser_share.png'), dpi=100, bbox_inches='tight')
plt.show()

## OS Version Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
os_counts = df['os'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(os_counts)))
os_counts.plot.barh(ax=ax, color=colors, edgecolor='black')
ax.set_title('Sessions by OS Version')
ax.set_xlabel('Sessions')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'os_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

# Legacy browser/OS risk
legacy = df[df['browser'].isin(['IE 11'])].shape[0]
print(f'\nLegacy browser (IE 11) sessions: {legacy} ({legacy/len(df)*100:.2f}%)')
old_android = df[df['os'] == 'Android 12'].shape[0]
print(f'Old Android 12 sessions: {old_android} ({old_android/len(df)*100:.2f}%)')

## Engagement by Device Type

In [ ]:
engagement = df.groupby('device_type').agg(
    avg_pages=('pages_viewed', 'mean'),
    median_duration=('duration_sec', 'median'),
    bounce_rate=('bounce', 'mean')
).round(3)
print('Engagement by device:')
print(engagement)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ['avg_pages', 'median_duration', 'bounce_rate'],
                           ['Avg Pages', 'Median Duration (s)', 'Bounce Rate']):
    engagement[col].plot.bar(ax=ax, edgecolor='black', color=['#2196F3', '#FF9800', '#4CAF50'])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_engagement.png'), dpi=100, bbox_inches='tight')
plt.show()

## Screen Resolution Analysis

In [ ]:
res_counts = df.groupby(['device_type', 'resolution']).size().reset_index(name='count')
res_counts = res_counts.sort_values(['device_type', 'count'], ascending=[True, False])
print('Top resolutions by device:')
for dev in device_types:
    sub = res_counts[res_counts['device_type'] == dev].head(4)
    print(f'\n  {dev}:')
    for _, row in sub.iterrows():
        pct = row['count'] / len(df[df['device_type'] == dev]) * 100
        print(f'    {row["resolution"]}: {pct:.1f}%')

## Interpretation and Business Insight
- **Mobile is growing** — mobile share is increasing ~0.5pp per month, projected to pass 50% soon
- **Chrome dominates** desktop and mobile with 50%+ combined share — prioritize Chrome testing
- **Safari (desktop+mobile)** is the second most important browser — Apple users are a large segment
- **IE 11** is <1% of traffic — safe to deprecate and save engineering effort
- **Desktop users** have 40% longer sessions and view more pages — higher engagement per session
- **Mobile bounce rate** is higher — mobile landing page optimization is needed
- **1920×1080** is the dominant desktop resolution — design breakpoints should target this first
- **Android 12** is ~1% of mobile traffic — candidate for deprecation

## Limitations
- Synthetic data with simplified distributions
- No geographic breakdown of device preferences
- No revenue attribution by device/browser
- No bot filtering (real analytics have 10-20% bot traffic)
- Browser version granularity is simplified

## How to Improve This Project
- Add geographic segmentation (device preferences vary by country)
- Include conversion rate by device/browser for revenue impact
- Add page load performance data per browser
- Build automated browser support matrix recommendations
- Include progressive web app vs native app comparison

## Production Considerations
- Automated browser support matrix updates (quarterly review)
- Real User Monitoring (RUM) for performance by device
- Responsive design testing across top 5 resolutions
- Deprecation communication plan for legacy browsers
- A/B testing of mobile-optimized vs responsive layouts

## Common Mistakes
- Supporting legacy browsers that represent <1% of traffic
- Designing for desktop first when mobile is 50%+ of traffic
- Not testing on actual devices (emulators miss real issues)
- Treating all Safari versions as the same (iOS Safari has unique quirks)
- Reporting device stats without engagement context

## Mini Challenge / Exercises
1. What percentage of sessions come from browser/OS combinations you'd consider 'modern'?
2. If you drop IE 11 support, how many engineering hours per month would you save (assume 10 hrs/month)?
3. Calculate the cost-per-session for each device type assuming $0.05 CPM.
4. Project the mobile vs desktop share 6 months forward using the current trend.

## Final Summary / Key Takeaways
- Device and browser analytics drive engineering prioritization decisions
- Mobile share is growing — mobile-first design is increasingly important
- Legacy browser deprecation frees engineering time with minimal user impact
- Engagement varies significantly by device — optimize experiences per platform
- Regular browser/OS audits prevent supporting dead platforms